<a href="https://colab.research.google.com/github/AngieVenta/AngieVenta/blob/main/RetoEmpleados.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Employee Attrition Prediction
### Data Preparation for Binary Classification Model

**Objective:** Evaluate, transform and select the most relevant features from an employee dataset
to build a suitable input for a machine learning model that predicts whether an employee
will leave the company or not.

## Requirements
This notebook was developed in Google Colab. To run it successfully:
1. Upload the file `empleadosRETO.csv` to your Colab session by clicking the
folder icon on the left panel and then the upload button.
2. Run all cells in order from top to bottom.
3. The final dataset `EmpleadosAttritionFinal.csv` will be saved in the Colab
session and can be downloaded from the same folder panel.

## 1. Setup

In [48]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA
import warnings

In [49]:
EmpleadosAttrition = pd.read_csv('empleadosRETO.csv')
print(f"Shape: {EmpleadosAttrition.shape}")
EmpleadosAttrition.head()

Shape: (400, 30)


,Age,BusinessTravel,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,EnvironmentSatisfaction,Gender,...,PercentSalaryHike,PerformanceRating,RelationshipSatisfaction,StandardHours,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsInCurrentRole,YearsSinceLastPromotion,Attrition
0,50,Travel_Rarely,Research & Development,1 km,2,Medical,1,997,4,Male,...,22,4,3,80,32,1,2,4,1,No
1,36,Travel_Rarely,Research & Development,6 km,2,Medical,1,178,2,Male,...,20,4,4,80,7,0,3,2,0,No
2,21,Travel_Rarely,Sales,7 km,1,Marketing,1,1780,2,Male,...,13,3,2,80,1,3,3,0,1,Yes
3,52,Travel_Rarely,Research & Development,7 km,4,Life Sciences,1,1118,2,Male,...,19,3,4,80,18,4,3,6,4,No
4,33,Travel_Rarely,Research & Development,15 km,1,Medical,1,582,2,Male,...,12,3,4,80,15,2,4,6,7,Yes


## 2. Data Cleaning

In [50]:
# These columns have no predictive value:
# EmployeeCount  -> all values are 1
# EmployeeNumber -> unique ID per employee
# Over18         -> all values are "Y"
# StandardHours  -> all values are 80
cols_to_drop = ['EmployeeCount', 'EmployeeNumber', 'Over18', 'StandardHours']
EmpleadosAttrition.drop(columns=cols_to_drop, inplace=True)

print(f"Shape after dropping irrelevant columns: {EmpleadosAttrition.shape}")
EmpleadosAttrition.head()

Shape after dropping irrelevant columns: (400, 26)


,Age,BusinessTravel,Department,DistanceFromHome,Education,EducationField,EnvironmentSatisfaction,Gender,JobInvolvement,JobLevel,...,OverTime,PercentSalaryHike,PerformanceRating,RelationshipSatisfaction,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsInCurrentRole,YearsSinceLastPromotion,Attrition
0,50,Travel_Rarely,Research & Development,1 km,2,Medical,4,Male,3,4,...,No,22,4,3,32,1,2,4,1,No
1,36,Travel_Rarely,Research & Development,6 km,2,Medical,2,Male,3,2,...,No,20,4,4,7,0,3,2,0,No
2,21,Travel_Rarely,Sales,7 km,1,Marketing,2,Male,3,1,...,No,13,3,2,1,3,3,0,1,Yes
3,52,Travel_Rarely,Research & Development,7 km,4,Life Sciences,2,Male,3,3,...,No,19,3,4,18,4,3,6,4,No
4,33,Travel_Rarely,Research & Development,15 km,1,Medical,2,Male,3,3,...,Yes,12,3,4,15,2,4,6,7,Yes


In [51]:
# Extract hiring year as integer from HiringDate (format M/D/YYYY)
EmpleadosAttrition['Year'] = (
    EmpleadosAttrition['HiringDate'].str.split('/').str[2].astype(int)
)

# Calculate years the employee has been at the company up to 2018
EmpleadosAttrition['YearsAtCompany'] = 2018 - EmpleadosAttrition['Year']

EmpleadosAttrition[['HiringDate', 'Year', 'YearsAtCompany']].head(10)

,HiringDate,Year,YearsAtCompany
0,06/06/2013,2013,5
1,12/25/2015,2015,3
2,2/14/2017,2017,1
3,7/29/2010,2010,8
4,10/07/2011,2011,7
5,10/17/1996,1996,22
6,1/16/2016,2016,2
7,11/28/2012,2012,6
8,8/25/2006,2006,12
9,1/21/2012,2012,6


In [52]:
# Rename original column that contains "km" as text
EmpleadosAttrition.rename(
    columns={'DistanceFromHome': 'DistanceFromHome_km'}, inplace=True
)

# Create new integer column by removing " km" suffix
EmpleadosAttrition['DistanceFromHome'] = (
    EmpleadosAttrition['DistanceFromHome_km']
    .str.replace(' km', '', regex=False)
    .astype(int)
)

EmpleadosAttrition[['DistanceFromHome_km', 'DistanceFromHome']].head()

,DistanceFromHome_km,DistanceFromHome
0,1 km,1
1,6 km,6
2,7 km,7
3,7 km,7
4,15 km,15


In [53]:
EmpleadosAttrition.drop(
    columns=['Year', 'HiringDate', 'DistanceFromHome_km'], inplace=True
)

print(f"Shape after cleanup: {EmpleadosAttrition.shape}")
EmpleadosAttrition.head()

Shape after cleanup: (400, 26)


,Age,BusinessTravel,Department,Education,EducationField,EnvironmentSatisfaction,Gender,JobInvolvement,JobLevel,JobRole,...,PerformanceRating,RelationshipSatisfaction,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsInCurrentRole,YearsSinceLastPromotion,Attrition,YearsAtCompany,DistanceFromHome
0,50,Travel_Rarely,Research & Development,2,Medical,4,Male,3,4,Research Director,...,4,3,32,1,2,4,1,No,5,1
1,36,Travel_Rarely,Research & Development,2,Medical,2,Male,3,2,Manufacturing Director,...,4,4,7,0,3,2,0,No,3,6
2,21,Travel_Rarely,Sales,1,Marketing,2,Male,3,1,Sales Representative,...,3,2,1,3,3,0,1,Yes,1,7
3,52,Travel_Rarely,Research & Development,4,Life Sciences,2,Male,3,3,Healthcare Representative,...,3,4,18,4,3,6,4,No,8,7
4,33,Travel_Rarely,Research & Development,1,Medical,2,Male,3,3,Manager,...,3,4,15,2,4,6,7,Yes,7,15


## 3. Data Analysis & Scaling

In [54]:
# Calculate average monthly income per department
SueldoPromedio = EmpleadosAttrition.groupby('Department')['MonthlyIncome'].mean()

# Create informational DataFrame with the results
SueldoPromedioDepto = SueldoPromedio.reset_index()
SueldoPromedioDepto.columns = ['Department', 'SueldoPromedio']

SueldoPromedioDepto

,Department,SueldoPromedio
0,Human Resources,6239.888889
1,Research & Development,6804.149813
2,Sales,7188.250000


In [55]:
# MonthlyIncome has much larger values compared to other variables
scaler = MinMaxScaler()
EmpleadosAttrition['MonthlyIncome'] = scaler.fit_transform(
    EmpleadosAttrition[['MonthlyIncome']]
)

print(f"Min: {EmpleadosAttrition['MonthlyIncome'].min():.4f}")
print(f"Max: {EmpleadosAttrition['MonthlyIncome'].max():.4f}")
EmpleadosAttrition['MonthlyIncome'].describe()

Min: 0.0000
Max: 1.0000


,MonthlyIncome
count,400.000000
mean,0.311196
std,0.258308
min,0.000000
25%,0.106939
50%,0.222711
75%,0.462745
max,1.000000


## 4. Encoding Categorical Variables

In [56]:
# Label encode all remaining categorical columns
categorical_cols = [
    'BusinessTravel', 'Department', 'EducationField',
    'Gender', 'JobRole', 'MaritalStatus', 'Attrition', 'OverTime'
]

for col in categorical_cols:
    EmpleadosAttrition[col] = pd.Categorical(EmpleadosAttrition[col]).codes

print(EmpleadosAttrition.dtypes)
EmpleadosAttrition.head()

Age                           int64
BusinessTravel                 int8
Department                     int8
Education                     int64
EducationField                 int8
EnvironmentSatisfaction       int64
Gender                         int8
JobInvolvement                int64
JobLevel                      int64
JobRole                        int8
JobSatisfaction               int64
MaritalStatus                  int8
MonthlyIncome               float64
NumCompaniesWorked            int64
OverTime                       int8
PercentSalaryHike             int64
PerformanceRating             int64
RelationshipSatisfaction      int64
TotalWorkingYears             int64
TrainingTimesLastYear         int64
WorkLifeBalance               int64
YearsInCurrentRole            int64
YearsSinceLastPromotion       int64
Attrition                      int8
YearsAtCompany                int64
DistanceFromHome              int64
dtype: object


,Age,BusinessTravel,Department,Education,EducationField,EnvironmentSatisfaction,Gender,JobInvolvement,JobLevel,JobRole,...,PerformanceRating,RelationshipSatisfaction,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsInCurrentRole,YearsSinceLastPromotion,Attrition,YearsAtCompany,DistanceFromHome
0,50,2,1,2,3,4,1,3,4,5,...,4,3,32,1,2,4,1,0,5,1
1,36,2,1,2,3,2,1,3,2,4,...,4,4,7,0,3,2,0,0,3,6
2,21,2,2,1,2,2,1,3,1,8,...,3,2,1,3,3,0,1,1,1,7
3,52,2,1,4,1,2,1,3,3,0,...,3,4,18,4,3,6,4,0,8,7
4,33,2,1,1,3,2,1,3,3,3,...,3,4,15,2,4,6,7,1,7,15


## 5. Feature Selection

In [57]:
# Compute absolute linear correlation of all variables against Attrition
correlations = EmpleadosAttrition.corr()['Attrition'].drop('Attrition').abs()

print("Absolute correlation with Attrition:")
print(correlations.sort_values(ascending=False).to_string())

Absolute correlation with Attrition:
OverTime                    0.324777
JobLevel                    0.214266
TotalWorkingYears           0.213329
Age                         0.212121
YearsInCurrentRole          0.203918
MonthlyIncome               0.194936
MaritalStatus               0.180404
YearsAtCompany              0.176001
JobInvolvement              0.166785
JobSatisfaction             0.164957
EnvironmentSatisfaction     0.124327
BusinessTravel              0.082899
JobRole                     0.078684
TrainingTimesLastYear       0.070884
YearsSinceLastPromotion     0.069000
PercentSalaryHike           0.060880
Education                   0.055531
Department                  0.054236
DistanceFromHome            0.052732
EducationField              0.051184
RelationshipSatisfaction    0.030945
Gender                      0.028839
WorkLifeBalance             0.021723
NumCompaniesWorked          0.009082
PerformanceRating           0.006471


In [58]:
# Select variables that meet the threshold, always keeping Attrition
selected_vars = correlations[correlations >= 0.1].index.tolist()
selected_vars.append('Attrition')

EmpleadosAttritionFinal = EmpleadosAttrition[selected_vars].copy()

print(f"Selected variables ({len(selected_vars)}): {selected_vars}")
print(f"Shape: {EmpleadosAttritionFinal.shape}")
EmpleadosAttritionFinal.head()

Selected variables (12): ['Age', 'EnvironmentSatisfaction', 'JobInvolvement', 'JobLevel', 'JobSatisfaction', 'MaritalStatus', 'MonthlyIncome', 'OverTime', 'TotalWorkingYears', 'YearsInCurrentRole', 'YearsAtCompany', 'Attrition']
Shape: (400, 12)


,Age,EnvironmentSatisfaction,JobInvolvement,JobLevel,JobSatisfaction,MaritalStatus,MonthlyIncome,OverTime,TotalWorkingYears,YearsInCurrentRole,YearsAtCompany,Attrition
0,50,4,3,4,4,0,0.864269,0,32,4,5,0
1,36,2,3,2,2,0,0.207340,0,7,2,3,0
2,21,2,3,1,2,2,0.088062,0,1,0,1,1
3,52,2,3,3,2,2,0.497574,0,18,6,8,0
4,33,2,3,3,3,1,0.664470,1,15,6,7,1


## 6. Principal Component Analysis (PCA)

In [59]:
# Scale full frame before PCA
scaler_pca = MinMaxScaler()
features_scaled = scaler_pca.fit_transform(EmpleadosAttritionFinal)

# Fit PCA with all components
pca = PCA()
pca.fit(features_scaled)

# Find minimum components needed to explain 80% variance
cumulative_variance = np.cumsum(pca.explained_variance_ratio_)
n_components = np.argmax(cumulative_variance >= 0.80) + 1

# Apply PCA with selected number of components
EmpleadosAttritionPCA = pca.transform(features_scaled)[:, :n_components]

print("Cumulative variance per component:")
for i, v in enumerate(cumulative_variance):
    print(f"  PC{i}: {v:.4f}")

print(f"\nMinimum components needed to explain 80% variance: {n_components}")

Cumulative variance per component:
  PC0: 0.2421
  PC1: 0.4286
  PC2: 0.5734
  PC3: 0.7044
  PC4: 0.7904
  PC5: 0.8514
  PC6: 0.9011
  PC7: 0.9496
  PC8: 0.9804
  PC9: 0.9903
  PC10: 0.9967
  PC11: 1.0000

Minimum components needed to explain 80% variance: 6


In [60]:
# Print shape and variance

print(f"Shape:{EmpleadosAttritionPCA.shape}")
print(f"Variance:{pca.explained_variance_ratio_.sum():.4f}")
# Add each component as C0, C1, ... to EmpleadosAttritionFinal
for i in range(n_components):
    EmpleadosAttritionFinal = EmpleadosAttritionFinal.assign(
        **{f'C{i}': EmpleadosAttritionPCA[:, i]}
    )

print(f"Final columns: {list(EmpleadosAttritionFinal.columns)}")
print(f"Final shape: {EmpleadosAttritionFinal.shape}")
print(f"n_components: {n_components}")
print(f"Cumulative at PC4: {cumulative_variance[4]:.4f}")
print(f"Cumulative at PC5: {cumulative_variance[5]:.4f}")
EmpleadosAttritionFinal.head()

Shape:(400, 6)
Variance:1.0000
Final columns: ['Age', 'EnvironmentSatisfaction', 'JobInvolvement', 'JobLevel', 'JobSatisfaction', 'MaritalStatus', 'MonthlyIncome', 'OverTime', 'TotalWorkingYears', 'YearsInCurrentRole', 'YearsAtCompany', 'Attrition', 'C0', 'C1', 'C2', 'C3', 'C4', 'C5']
Final shape: (400, 18)
n_components: 6
Cumulative at PC4: 0.7904
Cumulative at PC5: 0.8514


,Age,EnvironmentSatisfaction,JobInvolvement,JobLevel,JobSatisfaction,MaritalStatus,MonthlyIncome,OverTime,TotalWorkingYears,YearsInCurrentRole,YearsAtCompany,Attrition,C0,C1,C2,C3,C4,C5
0,50,4,3,4,4,0,0.864269,0,32,4,5,0,-0.815722,0.402593,0.395250,0.011833,0.369123,-0.002928
1,36,2,3,2,2,0,0.207340,0,7,2,3,0,-0.114889,-0.343228,-0.228746,-0.092647,-0.320506,-0.165801
2,21,2,3,1,2,2,0.088062,0,1,0,1,1,0.724073,-0.564883,-0.563928,-0.017652,0.403147,0.221202
3,52,2,3,3,2,2,0.497574,0,18,6,8,0,-0.430160,0.112146,-0.364186,-0.133260,-0.029493,0.361988
4,33,2,3,3,3,1,0.664470,1,15,6,7,1,0.655607,0.798972,-0.266123,0.331511,0.288848,0.013206


## 7. Export Final Dataset

In [61]:
# Confirm that PCA components are placed after Attrition
print(f"Current column order: {list(EmpleadosAttritionFinal.columns)}")

Current column order: ['Age', 'EnvironmentSatisfaction', 'JobInvolvement', 'JobLevel', 'JobSatisfaction', 'MaritalStatus', 'MonthlyIncome', 'OverTime', 'TotalWorkingYears', 'YearsInCurrentRole', 'YearsAtCompany', 'Attrition', 'C0', 'C1', 'C2', 'C3', 'C4', 'C5']


In [62]:
# Move Attrition to the last position so PCA components come after it naturally
# or move it before the PCA components — both are acceptable per the instructions
cols = [col for col in EmpleadosAttritionFinal.columns if col != 'Attrition'] + ['Attrition']
EmpleadosAttritionFinal = EmpleadosAttritionFinal[cols]

print(f"Final column order: {list(EmpleadosAttritionFinal.columns)}")

# Save final dataset
EmpleadosAttritionFinal.to_csv('EmpleadosAttritionFinal.csv', index=False)
print("File saved: EmpleadosAttritionFinal.csv")
print(f"Shape: {EmpleadosAttritionFinal.shape}")
EmpleadosAttritionFinal.head()

Final column order: ['Age', 'EnvironmentSatisfaction', 'JobInvolvement', 'JobLevel', 'JobSatisfaction', 'MaritalStatus', 'MonthlyIncome', 'OverTime', 'TotalWorkingYears', 'YearsInCurrentRole', 'YearsAtCompany', 'C0', 'C1', 'C2', 'C3', 'C4', 'C5', 'Attrition']
File saved: EmpleadosAttritionFinal.csv
Shape: (400, 18)


,Age,EnvironmentSatisfaction,JobInvolvement,JobLevel,JobSatisfaction,MaritalStatus,MonthlyIncome,OverTime,TotalWorkingYears,YearsInCurrentRole,YearsAtCompany,C0,C1,C2,C3,C4,C5,Attrition
0,50,4,3,4,4,0,0.864269,0,32,4,5,-0.815722,0.402593,0.395250,0.011833,0.369123,-0.002928,0
1,36,2,3,2,2,0,0.207340,0,7,2,3,-0.114889,-0.343228,-0.228746,-0.092647,-0.320506,-0.165801,0
2,21,2,3,1,2,2,0.088062,0,1,0,1,0.724073,-0.564883,-0.563928,-0.017652,0.403147,0.221202,1
3,52,2,3,3,2,2,0.497574,0,18,6,8,-0.430160,0.112146,-0.364186,-0.133260,-0.029493,0.361988,0
4,33,2,3,3,3,1,0.664470,1,15,6,7,0.655607,0.798972,-0.266123,0.331511,0.288848,0.013206,1
